# LIF Drift-Correction — Driver Notebook

Thin notebook that drives the pipeline defined in **`lif_drift_correction_pipeline.py`**.

Use this notebook to:
1. Inspect what's inside a `.lif` file
2. Process a single series and review its registration log + GIF inline before batch-running
3. Tune parameters (max drift voxels, σ, percentiles) on one series, then commit to batch
4. Run the full batch over multiple `.lif` files
5. (Optional) Re-process pre-extracted TIFF stacks (Mode B)

**Workflow**: the heavy logic lives in the `.py`. Edit functions there; this notebook only orchestrates and visualizes.

## 1. Setup

In [ ]:
# Auto-reload the .py whenever you edit and save it — no kernel restart needed.
%load_ext autoreload
%autoreload 2

In [ ]:
import json
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Image, display

from lif_pipeline_json import (
    # parsing
    inspect_single_lif,
    load_series_from_lif,
    load_series_from_tiff_dir,
    discover_tiff_series,
    # registration
    register_volume_slice_by_slice,
    # outputs
    save_series_outputs,
    make_sidebyside_gif,
    make_registered_mip_png,
    # orchestration
    process_series,
    run_pipeline_lif,
    run_pipeline_tiff,
    # display helpers
    normalize_for_view,
    CMAP_BLUE, CMAP_GREEN, CMAP_RED,
    # interactive selection (CLI helpers reused in notebook)
    _gui_select_lif_files,
    _gui_select_tiff_files,
    _derive_series_and_base_from_tiffs,
)

In [ ]:
# === Pick mode ===
# "lif"  -> GUI picker for one or more .lif files
# "tiff" -> GUI picker for one or more pre-extracted TIFF stacks
MODE = "lif"

CHANNEL_LABELS = ("DAPI", "Reference", "Target")
CHANNEL_ORDER  = (0, 1, 2)

REQUIRED_CHANNELS = 3
REQUIRED_Z        = 10

# Sequential drift correction has just one knob: the jump threshold.
# If |T(k) - T(k-1)| exceeds this, the slice is re-registered using
# T(k-1) as initialization (recovery, not discard) — mirrors stack_Reg.m.
JUMP_THRESHOLD_VOXELS  = 5.0
GIF_DURATION_PER_FRAME = 0.25  # seconds

if MODE == "lif":
    LIF_PATHS = _gui_select_lif_files()              # list[Path]
    BASE_DIR  = LIF_PATHS[0].parent
    LIF_FILES = [str(p.name) for p in LIF_PATHS]     # for run_pipeline_lif compat
    SERIES_DIRS = None
elif MODE == "tiff":
    TIFF_PATHS = _gui_select_tiff_files()
    SERIES_DIRS, BASE_DIR = _derive_series_and_base_from_tiffs(TIFF_PATHS)
    LIF_PATHS = None
    LIF_FILES = None
else:
    raise ValueError(f"Unknown MODE: {MODE!r}  (use 'lif' or 'tiff')")

print(f"\nMODE     : {MODE}")
print(f"BASE_DIR : {BASE_DIR}")


## 2. Inspect a `.lif` file

Print all series in a single `.lif` and pick the volumetric ones.

In [ ]:
# Inspect the first selected .lif file
assert MODE == "lif", "This cell only applies in MODE='lif'."

lif_path = LIF_PATHS[0]
series_list = inspect_single_lif(lif_path, verbose=True)

volumetric = [
    s for s in series_list
    if s["channels"] == REQUIRED_CHANNELS and s["z_slices"] >= REQUIRED_Z
]
print(f"\n{len(volumetric)} volumetric series matched filter "
      f"(channels={REQUIRED_CHANNELS}, z={REQUIRED_Z}).")
for s in volumetric:
    print(f"  [{s['index']}] {s['name']}")

## 3. Process ONE series and review before batch-running

This is the iterative-tuning loop. Edit `MAX_TRANSLATION_VOXELS` / `SMOOTHING_SIGMA` in the cell above, re-run this cell, and confirm the registration log and GIF look reasonable before proceeding.

In [ ]:
# Pick a volumetric series to test — change the index value to select.
test_series_info = volumetric[3]

series = load_series_from_lif(
    lif_path=lif_path,
    series_info=test_series_info,
    channel_order=CHANNEL_ORDER,
    channel_labels=CHANNEL_LABELS,
)

print(f"Loaded series: {series.series_name}")
for label, stack in series.stacks.items():
    print(f"  {label:<10s} shape={stack.shape} dtype={stack.dtype} "
          f"min={stack.min()} max={stack.max()}")

In [ ]:
out = process_series(
    series=series,
    base_dir=BASE_DIR,
    channel_labels=CHANNEL_LABELS,
    jump_threshold_voxels=JUMP_THRESHOLD_VOXELS,
    gif_duration_per_frame=GIF_DURATION_PER_FRAME,
    verbose=True,
)


In [ ]:
# Inspect per-slice transform log for this series.
print(json.dumps(out["transform_log"], indent=2))

In [ ]:
# Quick summary table + drift trajectory plot.
# Schema: each log entry has {z, tx, ty, reason}. The (tx, ty) here are the
# transform that mapped raw[k] onto registered[k-1] — sequential drift
# correction, no separate cumulative offset.
log = out["transform_log"]
zs       = [t["z"] for t in log]
tx_arr   = np.array([t["tx"] for t in log])
ty_arr   = np.array([t["ty"] for t in log])
reasons  = [t["reason"] for t in log]

# Per-slice change in transform — what the jump-detection logic compares
# against jump_threshold_voxels.
delta = np.zeros(len(log))
delta[1:] = np.hypot(np.diff(tx_arr), np.diff(ty_arr))

reason_color = {
    "ok":                       "tab:green",
    "anchor":                   "tab:gray",
    "ok_after_jump_recovery":   "tab:orange",
}
colors = [reason_color.get(r, "black") for r in reasons]

fig, axes = plt.subplots(1, 2, figsize=(13, 3.8))

# Left: per-slice change |T(k)-T(k-1)| with jump threshold overlaid
axes[0].bar(zs, delta, color=colors)
axes[0].axhline(JUMP_THRESHOLD_VOXELS, color="k", ls="--", lw=1,
                label=f"jump threshold = {JUMP_THRESHOLD_VOXELS}")
axes[0].set_xlabel("Z slice")
axes[0].set_ylabel("|T(k) - T(k-1)|  (voxels)")
axes[0].set_title("Inter-slice transform change (jump-detection input)")
axes[0].legend()

# Right: drift trajectory (tx, ty path)
axes[1].plot(tx_arr, ty_arr, "-o", color="tab:blue", lw=1, ms=4)
for k, (x, y) in enumerate(zip(tx_arr, ty_arr)):
    if k % max(1, len(zs) // 8) == 0:
        axes[1].annotate(str(k), (x, y), textcoords="offset points",
                         xytext=(4, 4), fontsize=8, color="dimgray")
axes[1].set_xlabel("tx (voxels)")
axes[1].set_ylabel("ty (voxels)")
axes[1].set_title(f"{series.series_name} — drift trajectory")
axes[1].set_aspect("equal", adjustable="datalim")
axes[1].axhline(0, color="lightgray", lw=0.5)
axes[1].axvline(0, color="lightgray", lw=0.5)

plt.tight_layout(); plt.show()

from collections import Counter
print("Reason counts:", Counter(reasons))
print(f"Final cumulative drift: tx={tx_arr[-1]:+.2f}, ty={ty_arr[-1]:+.2f}  "
      f"(|t|={np.hypot(tx_arr[-1], ty_arr[-1]):.2f})")


In [ ]:
# Display the side-by-side GIF inline.
display(Image(filename=str(out["gif_path"])))

In [ ]:
# The MIP is auto-displayed by `make_registered_mip_png` when called from a
# notebook (it calls plt.show()). The saved PNG is at:
print(out["mip_path"])

## 4. Once happy, run the full batch (Mode A: `.lif` files)

Iterates every series in every selected `.lif` that passes the channel + Z filter, registers it, writes raw + registered TIFFs, GIFs, MIPs, and metadata.

In [ ]:
# Mode A: batch over selected .lif files
assert MODE == "lif", "Use the Mode B cell below if you selected TIFF files."

run_pipeline_lif(
    lif_files=[str(p) for p in LIF_PATHS],   # absolute paths from the picker
    base_dir=BASE_DIR,
    required_channels=REQUIRED_CHANNELS,
    required_z=REQUIRED_Z,
    channel_order=CHANNEL_ORDER,
    channel_labels=CHANNEL_LABELS,
    jump_threshold_voxels=JUMP_THRESHOLD_VOXELS,
    verbose=True,  
    denoise_lines=True,
    denoise_detection_neighborhood=31,
    denoise_detection_threshold=4.0,
    denoise_min_punctum_size=5,
)


In [ ]:
# Mode B: process the TIFF series the user selected via the picker
assert MODE == "tiff", "Set MODE='tiff' and re-run the setup cell to use this."

print(f"Processing {len(SERIES_DIRS)} series under {BASE_DIR}:")
for s in SERIES_DIRS:
    print(" ", s)

matched = 0
for sdir in SERIES_DIRS:
    series = load_series_from_tiff_dir(sdir, channel_labels=CHANNEL_LABELS)
    if series is None:
        print(f"[SKIP] Could not load series from {sdir}")
        continue
    z = series.stacks[CHANNEL_LABELS[0]].shape[0]
    if z < REQUIRED_Z:
        print(f"[SKIP] {sdir.name} (z={z}, required>={REQUIRED_Z})")
        continue
    matched += 1
    process_series(
        series=series,
        base_dir=BASE_DIR,
        channel_labels=CHANNEL_LABELS,
        jump_threshold_voxels=JUMP_THRESHOLD_VOXELS,
        gif_duration_per_frame=GIF_DURATION_PER_FRAME,
        verbose=True,
    )
print(f"\n[INFO] Processed {matched} series.")


## 6. Ad-hoc inspection helpers

Drop-in cells for spot-checking individual series after the batch finishes.

In [ ]:
# Pick any processed series folder and load its metadata.
if MODE == "lif":
    processed_dir = BASE_DIR / LIF_PATHS[0].stem / volumetric[4]["name"]
else:
    processed_dir = SERIES_DIRS[0]

meta_path = processed_dir / "metadata.json"
with open(meta_path) as f:
    meta = json.load(f)

print("Series :", meta["series_name"])
print("Z      :", meta["z_slices"])
print("Scale  :", meta.get("scale"))
print("Reg    :", meta.get("registration", {}).get("transform"))

In [ ]:
# Show the saved side-by-side GIF for that series.
gif_files = list((processed_dir / "GIFS").glob("*.gif"))
for gp in gif_files:
    print(gp)
    display(Image(filename=str(gp)))

In [ ]:
# Show the saved MIP PNG for that series.
mip_files = list((processed_dir / "MIP").glob("*.png"))
for mp in mip_files:
    print(mp)
    display(Image(filename=str(mp)))

In [ ]:
import numpy as np
from skimage import io
from pathlib import Path

ser_dir = (BASE_DIR / "1_PSD Ms MA1046_Homer Rb SYSY"
           / "DIW_63x_MultiExR_2")
raw  = io.imread(str(ser_dir / "stacks" / "Reference_stack.tif"))
reg  = io.imread(str(ser_dir / "registered_stacks" / "Reference_stack_registered.tif"))

# Direct visual diff at problem slice
import matplotlib.pyplot as plt
z = 28

fig, axes = plt.subplots(1, 4, figsize=(20, 5))
axes[0].imshow(raw[0],  cmap="gray", vmin=0, vmax=np.percentile(raw[0], 99.5))
axes[0].set_title("raw Z=0 (anchor)")
axes[1].imshow(raw[z],  cmap="gray", vmin=0, vmax=np.percentile(raw[z], 99.5))
axes[1].set_title(f"raw Z={z}")
axes[2].imshow(reg[z],  cmap="gray", vmin=0, vmax=np.percentile(reg[z], 99.5))
axes[2].set_title(f"registered Z={z}")
# Overlay: raw Z=0 in red, registered Z=z in green — should align if registration works
overlay = np.zeros((*raw[0].shape, 3), dtype=np.float32)
overlay[..., 0] = raw[z]  / (raw[z].max() + 1e-9)
overlay[..., 1] = reg[z]  / (reg[z].max() + 1e-9)
axes[3].imshow(overlay)
axes[3].set_title(f"raw Z=0 (R) vs registered Z={z} (G)")
for a in axes: a.axis("off")
plt.tight_layout(); plt.show()

## 7. Scanner-line noise removal — preview and parameter tuning

Optional preprocessing step that removes horizontal scanner-line artifacts
from sparse fluorescence Z-stacks before drift correction.

**Algorithm.** For each Z-slice, count the nonzero-pixel density per row.
Flag rows whose density exceeds a 31-row local median by more than 4 × MAD.
On flagged rows, perform 8-connected component labeling on the slice's
binary nonzero mask, and zero out pixels belonging to components smaller
than 5 pixels. This removes scattered single-pixel scanner artifacts while
preserving real fluorescent puncta (which form larger multi-row blobs).

**Why this is safe.** The connectivity criterion protects rows that were
flagged because they happen to traverse puncta-rich regions — those rows
contain large connected components, which are kept. Only truly isolated
scanner pixels are removed. Bright biological signal that happens to
intersect a corrupted row is preserved by the same mechanism.

**Workflow.** Run the cell below to preview the filter on a single slice
of the currently-loaded `series` before running the batch with
`denoise_lines=True`. Tune `DETECTION_THRESHOLD` (lower = catch more lines),
`MIN_PUNCTUM_SIZE` (smaller = more aggressive removal), and
`DETECTION_NEIGHBORHOOD` (rows used to compute the local trend). When
satisfied, set `denoise_lines=True` in the batch cell (section 4) and
those parameters will be passed through to every series processed.

In [ ]:
# === Preview line-denoising on a single slice ================================
# Tunable parameters — match these to the batch settings if you find a good combo.
DETECTION_NEIGHBORHOOD = 31    # rows used to compute the local trend (odd)
DETECTION_THRESHOLD    = 4.0   # MADs above local trend → flagged
MIN_PUNCTUM_SIZE       = 5     # connected components smaller than this on a
                               # flagged row are zeroed
TEST_CHANNEL           = "Reference"
TEST_Z                 = None  # None -> middle slice; or set an int Z index

import numpy as np
import matplotlib.pyplot as plt
from lif_pipeline_json import denoise_scanner_lines_2d

stack = series.stacks[TEST_CHANNEL]
z = TEST_Z if TEST_Z is not None else stack.shape[0] // 2
img = stack[z]

cleaned, info = denoise_scanner_lines_2d(
    img,
    detection_neighborhood=DETECTION_NEIGHBORHOOD,
    detection_threshold=DETECTION_THRESHOLD,
    min_punctum_size=MIN_PUNCTUM_SIZE,
)
print(f"Slice: {series.series_name} | {TEST_CHANNEL} | Z={z}")
print(f"  Bad rows detected:  {info['n_bad_rows']}")
print(f"  Pixels zeroed:      {info['n_pixels_zeroed']}")
print(f"  Flagged rows:       {info['bad_rows']}")

img_f = img.astype(np.float32)
cleaned_f = cleaned.astype(np.float32)
diff = cleaned_f - img_f

vmin, vmax = np.percentile(img_f, 1), np.percentile(img_f, 99.5)
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle(
    f"Line denoising preview — {series.series_name} | {TEST_CHANNEL} | Z={z}\n"
    f"NEIGHBORHOOD={DETECTION_NEIGHBORHOOD}, THRESHOLD={DETECTION_THRESHOLD} MAD, "
    f"MIN_PUNCTUM_SIZE={MIN_PUNCTUM_SIZE}",
    fontsize=12,
)

axes[0].imshow(img_f, cmap="gray", vmin=vmin, vmax=vmax)
axes[0].set_title("Original")
axes[0].axis("off")

axes[1].imshow(cleaned_f, cmap="gray", vmin=vmin, vmax=vmax)
axes[1].set_title(f"Cleaned ({info['n_pixels_zeroed']} px zeroed on "
                  f"{info['n_bad_rows']} rows)")
axes[1].axis("off")

dmax = max(np.percentile(np.abs(diff), 99), 1.0)
axes[2].imshow(diff, cmap="RdBu_r", vmin=-dmax, vmax=dmax)
axes[2].set_title(f"Difference (cleaned - original), ±{dmax:.1f}")
axes[2].axis("off")

plt.tight_layout()
plt.show()

print()
print("Tuning hints:")
print("  - If lines are visible but NOT detected: lower DETECTION_THRESHOLD")
print("  - If clean rows are being modified: raise DETECTION_THRESHOLD")
print("  - If line detected but pixels not zeroed: raise MIN_PUNCTUM_SIZE")
print("  - If real puncta on flagged rows are removed: lower MIN_PUNCTUM_SIZE")
print()
print("To enable in the batch run, edit section 4's run_pipeline_lif call to add:")
print(f"  denoise_lines=True,")
print(f"  denoise_detection_neighborhood={DETECTION_NEIGHBORHOOD},")
print(f"  denoise_detection_threshold={DETECTION_THRESHOLD},")
print(f"  denoise_min_punctum_size={MIN_PUNCTUM_SIZE},")